In [ ]:
import numpy as np
import MDAnalysis as mda
import os
import pandas as pd
import pickle
import matplotlib.pyplot as plt

from code_libraries.counts_rdfs_3 import *
from code_libraries.predict_acyl_chains_NN_by_chain import *
from scipy import integrate

In [ ]:
def convolve_positions(pos_array, n_point):

    n_frames, n_lipids, n_coords = pos_array.shape
    n_reduced = n_frames - n_point + 1
    kernel = np.ones(n_point) / n_point

    convolved = np.empty([n_reduced, n_lipids, n_coords])

    for j in range(n_lipids):
        for c in range(n_coords):
            convolved[:, j, c] = np.convolve(pos_array[:, j, c], kernel, mode="valid")

    return convolved

In [ ]:
start_frame , stop_frame = 0 , 2000
sys_type = "sym"
sys_name_75 = "dopc_75"


sn1_indices = [ 41,  42,  43,  91,  92,  93,  75,  95,  96,  97,  98,  99, 100,
   101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113,
   114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126,
   127, 128, 129, 130, 131, 132, 133]

sn2_indices = [32, 33, 34, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57,
   58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74,
   75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86]



In [ ]:
data_array_dir = "../heavy_data_NO_GIT/data_arrays/"

In [ ]:
dopc_75_box_dim = np.load(os.path.join(data_array_dir , "dopc_75_sn1_ft_last_11pt_box_dim.npy"))
dopc_75_11pt_sn1_prediction_labels = np.load(os.path.join(data_array_dir , "dopc_75_sn1_ft_last_11pt_prediction_labels.npy") , allow_pickle=True)
dopc_75_11pt_sn2_prediction_labels = np.load(os.path.join(data_array_dir , "dopc_75_sn2_ft_last_11pt_prediction_labels.npy") , allow_pickle=True)
dopc_75_box_dim = dopc_75_box_dim[: , :2]
# dopc_75_11pt_sn1_prediction_labels = np.hstack([dopc_75_11pt_sn1_prediction_labels[0] , dopc_75_11pt_sn1_prediction_labels[1]])
# dopc_75_11pt_sn2_predcition_labels = np.hstack([dopc_75_11pt_sn2_prediction_labels[0] , dopc_75_11pt_sn2_prediction_labels[1]])

sys_75_box_dim = dopc_75_box_dim
sys_75_box_dim = sys_75_box_dim[: , :2]
sys_75_box_dim = np.mean(sys_75_box_dim , axis = 0)

In [ ]:
dopc_75_11pt_sn1_pred_labels_lower  , dopc_75_11pt_sn1_pred_labels_upper = dopc_75_11pt_sn1_prediction_labels
dopc_75_11pt_sn2_pred_labels_lower  , dopc_75_11pt_sn2_pred_labels_upper = dopc_75_11pt_sn2_prediction_labels


In [ ]:
dopc_75_11pt_sn1_pred_labels_lower.shape

In [ ]:
dopc_75_pos_raw = clf_lo_ld_by_chain(sys_name_75 , sys_type , "raw" , "nn" , "sn1" , 3 , start_frame , stop_frame).bilayer_all_dopc_data()
chol_75_pos_raw = clf_lo_ld_by_chain(sys_name_75 , sys_type , "raw" , "nn" , "sn1", 3 , start_frame , stop_frame).bilayer_all_chol_data()

In [ ]:
dopc_75_pos_lower , dopc_75_pos_upper = dopc_75_pos_raw[0] , dopc_75_pos_raw[1]
chol_75_pos_lower , chol_75_pos_upper = chol_75_pos_raw[0] , chol_75_pos_raw[1]

dopc_75_phos_lower = dopc_75_pos_lower[: , : , 19 , :]
dopc_75_phos_upper = dopc_75_pos_upper[: , : , 19 , :]
dopc_75_nitro_lower = dopc_75_pos_lower[: , : , 0 , :]
dopc_75_nitro_upper = dopc_75_pos_lower[: , : , 0 , :]
chol_75_oxy_lower = chol_75_pos_lower[: , : , 2 , :]
chol_75_oxy_upper = chol_75_pos_upper[: , : , 2 , :]

dopc_75_phos_lower_11pt = convolve_positions(dopc_75_phos_lower , 11)
dopc_75_phos_upper_11pt = convolve_positions(dopc_75_phos_upper , 11)
dopc_75_nitro_lower_11pt = convolve_positions(dopc_75_nitro_lower ,11)
dopc_75_nitro_upper_11pt = convolve_positions(dopc_75_nitro_upper , 11)
chol_75_oxy_lower_11pt = convolve_positions(chol_75_oxy_lower , 11)
chol_75_oxy_upper_11pt = convolve_positions(chol_75_oxy_upper , 11)

In [ ]:
pbc_distances_75_phos_chol_lower = compute_distances_PBC_acyl_to_chol_all_ts(dopc_75_phos_lower_11pt , chol_75_oxy_lower_11pt , sys_75_box_dim)
pbc_distances_75_phos_chol_upper = compute_distances_PBC_acyl_to_chol_all_ts(dopc_75_phos_upper_11pt , chol_75_oxy_upper_11pt , sys_75_box_dim)
pbc_distances_75_nitro_chol_lower = compute_distances_PBC_acyl_to_chol_all_ts(dopc_75_nitro_lower_11pt , chol_75_oxy_lower_11pt , sys_75_box_dim)
pbc_distances_75_nitro_chol_upper = compute_distances_PBC_acyl_to_chol_all_ts(dopc_75_nitro_upper_11pt , chol_75_oxy_upper_11pt , sys_75_box_dim)

In [ ]:
indices_75_11pt_both_lo_lower = dopc_75_11pt_sn1_pred_labels_lower & dopc_75_11pt_sn2_pred_labels_lower
indices_75_11pt_both_lo_upper = dopc_75_11pt_sn1_pred_labels_upper & dopc_75_11pt_sn2_pred_labels_upper

indices_75_11pt_both_ld_lower = ~dopc_75_11pt_sn1_pred_labels_lower & ~dopc_75_11pt_sn2_pred_labels_lower
indices_75_11pt_both_ld_upper = ~dopc_75_11pt_sn1_pred_labels_upper & ~dopc_75_11pt_sn2_pred_labels_upper


In [ ]:
both_lo_pbc_distances_phos_chol_75_11pt_lower = predictions_cutoff_pbc(pbc_distances_75_phos_chol_lower , indices_75_11pt_both_lo_lower)[0]
both_lo_pbc_distances_phos_chol_75_11pt_upper = predictions_cutoff_pbc(pbc_distances_75_phos_chol_upper , indices_75_11pt_both_lo_upper)[0]

both_ld_pbc_distances_phos_chol_75_11pt_lower = predictions_cutoff_pbc(pbc_distances_75_phos_chol_lower , indices_75_11pt_both_ld_lower)[0]
both_ld_pbc_distances_phos_chol_75_11pt_upper = predictions_cutoff_pbc(pbc_distances_75_phos_chol_upper , indices_75_11pt_both_ld_upper)[0]


both_lo_pbc_distances_nitro_chol_75_11pt_lower = predictions_cutoff_pbc(pbc_distances_75_nitro_chol_lower , indices_75_11pt_both_lo_lower)[0]
both_lo_pbc_distances_nitro_chol_75_11pt_upper = predictions_cutoff_pbc(pbc_distances_75_nitro_chol_upper , indices_75_11pt_both_lo_upper)[0]

both_ld_pbc_distances_nitro_chol_75_11pt_lower = predictions_cutoff_pbc(pbc_distances_75_nitro_chol_lower , indices_75_11pt_both_ld_lower)[0]
both_ld_pbc_distances_nitro_chol_75_11pt_upper = predictions_cutoff_pbc(pbc_distances_75_nitro_chol_upper , indices_75_11pt_both_ld_upper)[0]




In [ ]:
dr = 0.05
radius = 2

radial_distances_75 = np.arange(0 , radius , dr)

both_lo_phos_chol_rdf_75_11pt_lower = RDF_over_all_frames_diff_species(both_lo_pbc_distances_phos_chol_75_11pt_lower , dr , radius ,sys_75_box_dim )
both_lo_phos_chol_rdf_75_11pt_upper = RDF_over_all_frames_diff_species(both_lo_pbc_distances_phos_chol_75_11pt_upper , dr , radius ,sys_75_box_dim )

both_ld_phos_chol_rdf_75_11pt_lower = RDF_over_all_frames_diff_species(both_ld_pbc_distances_phos_chol_75_11pt_lower , dr , radius ,sys_75_box_dim )
both_ld_phos_chol_rdf_75_11pt_upper = RDF_over_all_frames_diff_species(both_ld_pbc_distances_phos_chol_75_11pt_upper , dr , radius ,sys_75_box_dim )


both_lo_nitro_chol_rdf_75_11pt_lower = RDF_over_all_frames_diff_species(both_lo_pbc_distances_nitro_chol_75_11pt_lower , dr , radius ,sys_75_box_dim )
both_lo_nitro_chol_rdf_75_11pt_upper = RDF_over_all_frames_diff_species(both_lo_pbc_distances_nitro_chol_75_11pt_upper , dr , radius ,sys_75_box_dim )

both_ld_nitro_chol_rdf_75_11pt_lower = RDF_over_all_frames_diff_species(both_ld_pbc_distances_nitro_chol_75_11pt_lower , dr , radius ,sys_75_box_dim )
both_ld_nitro_chol_rdf_75_11pt_upper = RDF_over_all_frames_diff_species(both_ld_pbc_distances_nitro_chol_75_11pt_upper , dr , radius ,sys_75_box_dim )


av_75_11pt_phos_chol_both_lo_rdf = 0.5*(np.array(both_lo_phos_chol_rdf_75_11pt_lower) + np.array(both_lo_phos_chol_rdf_75_11pt_upper))
av_75_11pt_phos_chol_both_ld_rdf = 0.5*(np.array(both_ld_phos_chol_rdf_75_11pt_lower) + np.array(both_ld_phos_chol_rdf_75_11pt_upper))

av_75_11pt_nitro_chol_both_lo_rdf = 0.5*(np.array(both_lo_nitro_chol_rdf_75_11pt_lower) + np.array(both_lo_nitro_chol_rdf_75_11pt_upper))
av_75_11pt_nitro_chol_both_ld_rdf = 0.5*(np.array(both_ld_nitro_chol_rdf_75_11pt_lower) + np.array(both_ld_nitro_chol_rdf_75_11pt_upper))

In [ ]:
dr = 0.05
radius = 2

radial_distances_75 = np.arange(0 , radius , dr)

In [ ]:
plt.rcParams['font.weight'] = 'bold'
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['axes.titleweight'] = 'bold'

fig, ax = plt.subplots(figsize=(14, 10))
#ax.set_title(r'Headgroup-CHOL RDF, $\mathbf{\chi_C}$ = 0.25', fontsize=30, fontweight='bold')
ax.plot(radial_distances_75, av_75_11pt_phos_chol_both_lo_rdf, color="green", linewidth=3, label="CHOL(O)-DOPC(P)")
ax.plot(radial_distances_75, av_75_11pt_phos_chol_both_ld_rdf, color="blue", linewidth=3, label="CHOL(O)-DOPC(N)")
ax.plot(radial_distances_75, av_75_11pt_nitro_chol_both_lo_rdf, color="red", linewidth=3, label="DOPC(P)-DOPC(P)")
ax.plot(radial_distances_75, av_75_11pt_nitro_chol_both_ld_rdf, color="black", linewidth=3, label="DOPC(P)-DOPC(N)")

ax.axhline(y=1, color='gray', linestyle=':', linewidth=2)
# ax.text(0.05, 0.95, r'$\mathbf{\chi_C = 0.25}$', transform=ax.transAxes, fontsize=25, fontweight='bold',
#         verticalalignment='top', horizontalalignment='left')
ax.set_xlabel(r'$\mathbf{r \, (nm)}$', fontsize=32, fontweight='bold')
ax.set_ylabel(r'$\mathbf{g(r)}$', fontsize=32, fontweight='bold')
ax.set_xlim(0, 2.0)
ax.set_ylim(0, 1.5)
ax.set_xticks([0.5, 1.0, 1.5, 2.0])
ax.set_yticks([0.0, 0.3, 0.6, 0.9, 1.2, 1.5])
ax.tick_params(axis='both', which='major', labelsize=32, width=2.5, length=8, direction="inout")
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontweight('bold')
legend = ax.legend(fontsize=20, prop={'weight':'bold', 'size': 30}, frameon=True, fancybox=True,
                  loc = "lower right")
for line in legend.get_lines():
    line.set_linewidth(3.0)
plt.tight_layout()
plt.show()

In [ ]:
av_75_11pt_phos_chol_both_lo_rdf = 0.5*(np.array(both_lo_phos_chol_rdf_75_11pt_lower) + np.array(both_lo_phos_chol_rdf_75_11pt_upper))
av_75_11pt_phos_chol_both_ld_rdf = 0.5*(np.array(both_ld_phos_chol_rdf_75_11pt_lower) + np.array(both_ld_phos_chol_rdf_75_11pt_upper))

av_75_11pt_nitro_chol_both_lo_rdf = 0.5*(np.array(both_lo_nitro_chol_rdf_75_11pt_lower) + np.array(both_lo_nitro_chol_rdf_75_11pt_upper))
av_75_11pt_nitro_chol_both_ld_rdf = 0.5*(np.array(both_ld_nitro_chol_rdf_75_11pt_lower) + np.array(both_ld_nitro_chol_rdf_75_11pt_upper))

In [ ]:
integrate.trapezoid(av_75_11pt_nitro_chol_both_lo_rdf ,radial_distances_75 ) - integrate.trapezoid(av_75_11pt_phos_chol_both_lo_rdf ,radial_distances_75 )

In [ ]:
integrate.trapezoid(av_75_11pt_nitro_chol_both_ld_rdf ,radial_distances_75 ) - integrate.trapezoid(av_75_11pt_phos_chol_both_ld_rdf ,radial_distances_75 )

In [ ]:
abs_diff_75_11pt_lo_umb_index = np.abs(av_75_11pt_nitro_chol_both_lo_rdf - av_75_11pt_phos_chol_both_lo_rdf)
abs_diff_75_11pt_ld_umb_index = np.abs(av_75_11pt_nitro_chol_both_ld_rdf - av_75_11pt_phos_chol_both_ld_rdf)

# abs_diff_75_11pt_lo_umb_index = (av_75_11pt_nitro_chol_both_lo_rdf - av_75_11pt_phos_chol_both_lo_rdf)
# abs_diff_75_11pt_ld_umb_index = (av_75_11pt_nitro_chol_both_ld_rdf - av_75_11pt_phos_chol_both_ld_rdf)

In [ ]:
end_index = 40

integrate.trapezoid(abs_diff_75_11pt_lo_umb_index[:end_index] ,radial_distances_75[:end_index]) , integrate.trapezoid(abs_diff_75_11pt_ld_umb_index[:end_index] , radial_distances_75[:end_index])